# s18 — Event Bus

**What this teaches:** how to subscribe to the event stream that every agent emits internally. The `core/events` package gives you a typed bus, a file logger, and a counting handler — register them once and you get a structured trace of `session_start`, `before_model`, `after_model`, `before_tool`, `after_tool`, `tool_error`, `run_start`, `run_end`, and `session_end` events for free.

**Concept anchor:** the agent loop emits a fixed vocabulary of lifecycle events at well-known points (before/after each model call, before/after each tool call, on session start/end). Plugins attach handlers to those events — that is exactly how compression, permissions, and the cache stats plugin observe the loop without modifying it.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- Any configured LLM provider.
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```
- The notebook writes to `./logs/agent_events_<ts>.log`. Make sure the working directory is writable.

## 1. Imports

Three additions vs. [s01_loop](../s01_loop/s01_loop.ipynb): `core/events` (bus + helpers), `core/tools` (so the agent actually emits tool events), and `path/filepath`/`time` for the log filename.

In [ ]:
import (
    "context"
    "fmt"
    "os"
    "path/filepath"
    "time"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/events"
    "github.com/blouargant/yoke/core/stream"
    fstools "github.com/blouargant/yoke/core/tools"
)

## 2. Helper

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the bus, file logger, and counter

`events.NewBus()` is the dispatcher. `events.FileLogger(path)` returns a handler that writes one JSON line per event. `events.NewCounter()` returns a stats struct plus a handler that increments counters per event type.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
bus := events.NewBus()
must(os.MkdirAll("logs", 0o755))
logName := "agent_events_" + time.Now().Format("20060102_150405") + ".log"
logger, closeLog, err := events.FileLogger(filepath.Join("logs", logName))
must(err)
defer closeLog()
counter, counterH := events.NewCounter()

## 4. Subscribe both handlers to every interesting event

We register the *same* two handlers (`logger`, `counterH`) against every event in the canonical list. The full vocabulary you'll see flowing past:

- `session_start` / `session_end` — lifecycle of a `(userID, sessionID)` pair
- `run_start` / `run_end` — boundaries of a single `RunOnce`/`Run`
- `before_model` / `after_model` — every model invocation in the loop
- `before_tool` / `after_tool` — every tool dispatch in the loop
- `tool_error` — fired when a tool returns an error
- `curate_now` — emitted when curation is requested (idle harvest, session end, manual)

In [ ]:
for _, ev := range []string{
    events.EventBeforeTool, events.EventAfterTool,
    events.EventBeforeModel, events.EventAfterModel,
    events.EventToolError,
    events.EventSessionStart, events.EventSessionEnd,
    events.EventRunStart, events.EventRunEnd,
    events.EventCurateNow,
} {
    bus.On(ev, logger).On(ev, counterH)
}
plug, err := bus.Plugin("events")
must(err)

## 5. Run the loop with the events plugin attached

Tools are wired in so the loop actually produces `before_tool`/`after_tool` events for the counter. After the run, we print the counter summary — and you can `cat logs/agent_events_*.log` to see the full structured trace the file logger wrote.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:  "s18_events",
    Model: llm,
    Tools: fstools.New(),
})
must(err)
r, err := agentkit.Runner("s18", a, plug)
must(err)
prompt := "List the files in the current directory using bash."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))
fmt.Println("\nEvent counts:")
fmt.Print(counter.Summary())

## What to look for

- The counter prints something like `before_model: 2, after_model: 2, before_tool: 1, after_tool: 1, …` — you can read the whole shape of the run off the counts alone.
- The JSON-lines file under `logs/` is the same format other internal plugins consume — pair with [s15_parallel](../s15_parallel/s15_parallel.ipynb) to see multiple `before_tool` events between a single pair of `before_model`/`after_model` events, and with [s20_compress](../s20_compress/s20_compress.ipynb) to see the audit trail of compressions firing.

## Try it yourself

1. Remove `events.EventBeforeTool` and `events.EventAfterTool` from the subscription loop and rerun — the counter summary loses the tool rows, even though the tools still run.
2. Add a custom handler with `bus.On(events.EventAfterTool, func(ctx, ev) error { fmt.Println("tool:", ev) ; return nil })` to print every tool dispatch inline.